# nanostat: learn agent RL with verifiable rewards

A tiny GPT (about 100k parameters) learns to **plan** a hypothesis test. Deterministic code runs the test; the model never computes a number.
It is trained with SFT and then GRPO, using two different verifiable rewards at the same compute.

Each section below runs one lesson. Read the lesson text on the [course site](https://anandharajan.github.io/nanostat-ctv/).
Everything runs on the free CPU runtime; a GPU is not needed. Synthetic data only, for learning.

In [ ]:
!git clone -q https://github.com/Anandharajan/nanostat-ctv
%cd nanostat-ctv
!pip -q install numpy scipy torch   # already present on Colab; this is a no-op there

## 1 · Tests from scratch
[Read lesson](https://anandharajan.github.io/nanostat-ctv/lessons/01-statistics.html), then run:

In [ ]:
!python edu/nanostat.py stats

## 2 · Simulate the null
[Read lesson](https://anandharajan.github.io/nanostat-ctv/lessons/02-null-simulation.html), then run:

In [ ]:
!python edu/nanostat.py null

## 3 · Tasks, answer keys, typed plans
[Read lesson](https://anandharajan.github.io/nanostat-ctv/lessons/03-tasks-and-plans.html), then run:

In [ ]:
!python edu/nanostat.py tasks

## 4 · Rewards and reward hacking
[Read lesson](https://anandharajan.github.io/nanostat-ctv/lessons/04-rewards.html), then run:

In [ ]:
!python edu/nanostat.py rewards

## 5 · A tiny GPT
[Read lesson](https://anandharajan.github.io/nanostat-ctv/lessons/05-tiny-gpt.html), then run:

In [ ]:
!python edu/nanostat.py model

## 6 · Supervised fine-tuning
[Read lesson](https://anandharajan.github.io/nanostat-ctv/lessons/06-sft.html), then run:

In [ ]:
!python edu/nanostat.py sft

## 7 · GRPO (about 1 minute per run)
[Read lesson](https://anandharajan.github.io/nanostat-ctv/lessons/07-grpo.html), then run:

In [ ]:
!python edu/nanostat.py grpo --reward ctv
!python edu/nanostat.py grpo --reward fisher

## 8 · The experiment (about 6 minutes)
[Read lesson](https://anandharajan.github.io/nanostat-ctv/lessons/08-the-experiment.html), then run:

In [ ]:
!python edu/nanostat.py compare

## Hack it yourself

Import the module and change things live. For example, lesson 4's reward hack: set `MALFORMED = 0.0` and retrain.
After an exercise, restart the runtime or re-import, so that you start fresh.

In [ ]:
import importlib
import sys

import numpy as np

sys.path.insert(0, "edu")
import nanostat as ns

importlib.reload(ns)

ns.MALFORMED = 0.0  # try -0.5 (the default) vs 0.0
vocab = ns.build_vocab()
rng = np.random.default_rng(0)
train = [ns.make_task(rng) for _ in range(2000)]
test = [ns.make_task(np.random.default_rng(10_000 + i)) for i in range(600)]
agent = ns.Agent(vocab)
ns.sft(agent, train[:40])
ns.grpo(agent, train, "ctv", rollouts=12800)
plans = agent.decode(agent.act([agent.encode(t["obs"]) for t in test], greedy=True))
broken = sum(ns.execute(t, p) is None for t, p in zip(test, plans))
print("broken plans:", broken, "/", len(test), "| eval:", ns.evaluate(agent, test))

## 9 · From nano to real
[Read lesson 9](https://anandharajan.github.io/nanostat-ctv/lessons/09-next-steps.html): how each piece maps to the full NanoStat-CTV system.